In [ ]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("../config/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 31))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


In [ ]:
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from scripts.weighted_coverage import wknn
from scripts.utils import dataset_tp_rp_split, extract_unique_npcis
from scripts.weighted_coverage import create_point_matrix, compute_weights
import pandas as pd


def compute_weights_pca(m_rfp_pca, m_tp_pca):
    """
    Compute weights using PCA-transformed data
    """
    # Compute Euclidean distances in the PCA space
    D = cdist(m_tp_pca, m_rfp_pca, metric="euclidean")

    # Sort distances and compute weights
    idx_sort = np.argsort(D, axis=1)
    D_sort = np.take_along_axis(D, idx_sort, axis=1)

    # Avoid division by zero
    min_nonzero_distance = np.min(D[D > 0]) if np.any(D > 0) else 0.1
    D_sort[D_sort == 0] = min_nonzero_distance / 20

    W = 1.0 / D_sort
    return W, idx_sort


def process_test_points_pca(
        df_tp: pd.DataFrame,
        df_rp: pd.DataFrame,
        pcis: list[tuple],
        rf_param: RF_PARAM_5G,
        k: int = 2,
        n_components: int = 0.95,
):
    # 1. Create the full point matrix with all beam features
    m_rp_full, idx_rp_full = create_point_matrix(df_rp, pcis, rf_param)
    m_tp_full, idx_tp_full = create_point_matrix(df_tp, pcis, rf_param)

    # 3. Apply PCA to reduce dimensions
    pca = PCA(n_components=n_components)
    pca.fit(m_rp_full)
    m_rp_pca = pca.transform(m_rp_full)
    m_tp_pca = pca.transform(m_tp_full)

    W_pca, idx_sort_pca = compute_weights_pca(m_rp_pca, m_tp_pca)

    W, idx_sort = compute_weights(m_rp_full, idx_rp_full, m_tp_full, idx_tp_full)

    _, errors = wknn(df_tp, df_rp, idx_sort_pca, W_pca, k)

    _, errors_control = wknn(df_tp, df_rp, idx_sort, W, k=2)

    n_points = errors.shape[0]

    complexity = np.repeat(m_rp_pca.shape[0] * m_tp_pca.shape[1], n_points)

    complexity_control = np.repeat(m_rp_full.shape[0] * m_tp_full.shape[1], n_points)

    res = np.array([
        errors,
        complexity,
        errors_control,
        complexity_control,
    ])

    return res.T




In [3]:
df = df_orig.sample(3000)
all_pcis = extract_unique_npcis(df['measurements_matrix'])

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 420 * 69)
data = process_test_points_pca(df_tp, df_rp, all_pcis, rf_param=rf_param, k=2)

res_df = pd.DataFrame(data, columns=['error', 'complexity', 'error_control', 'complexity_control'])

print('mean', res_df.mean())

res_df

mean error                      7.592794
complexity            165360.000000
error_control              7.394602
complexity_control    551200.000000
dtype: float64


,error,complexity,error_control,complexity_control
0,1.588515,165360.0,1.477271,551200.0
1,9.476576,165360.0,10.298743,551200.0
2,0.238784,165360.0,0.234648,551200.0
3,0.229029,165360.0,0.231022,551200.0
4,0.118612,165360.0,0.079600,551200.0
...,...,...,...,...
875,0.938046,165360.0,0.867308,551200.0
876,3.440680,165360.0,3.053378,551200.0
877,15.514286,165360.0,8.911122,551200.0
878,1.002810,165360.0,1.280391,551200.0


In [4]:
res_df.to_csv('pca_data_2.csv')

In [5]:
data

array([[1.58851534e+00, 1.65360000e+05, 1.47727119e+00, 5.51200000e+05],
       [9.47657625e+00, 1.65360000e+05, 1.02987434e+01, 5.51200000e+05],
       [2.38784242e-01, 1.65360000e+05, 2.34648100e-01, 5.51200000e+05],
       ...,
       [1.55142862e+01, 1.65360000e+05, 8.91112202e+00, 5.51200000e+05],
       [1.00281002e+00, 1.65360000e+05, 1.28039060e+00, 5.51200000e+05],
       [5.00002209e+01, 1.65360000e+05, 2.17378928e+01, 5.51200000e+05]])

In [6]:
print(f"""
error {res_df['error'].mean():.2f}
complexity {res_df['complexity'].mean() / 1000:.0f} K

error control {res_df['error_control'].mean():.2f}
complexity control {res_df['complexity_control'].mean() / 1000:.0f} k
""")


error 7.59
complexity 165 K

error control 7.39
complexity control 551 k

